# MAGFiLO Solar Filament Segmentation - EDA

This notebook examines the MAGFiLO 1.0 dataset before modelling. It treats the supplied COCO-style JSON as the source of annotation truth, then checks how those records map to the local JPEG files. The analysis focuses on annotation coverage, morphology categories, filament geometry, image intensity, data quality, and visual overlays.

> **Important:** the annotation JSON may contain more image records than the local train/test folders. Counts are therefore reported separately for annotation records, local files, and matched files. A missing local file is a coverage issue, not evidence that its annotation is invalid.

## Objectives

We will answer six practical questions:

1. How many images and filament instances are available, and how completely do the JSON records map to local files?
2. How are the four morphology categories distributed across images and instances?
3. What are the typical filament sizes, shapes, aspect ratios, occupied image areas, and spine lengths?
4. Are polygons, bounding boxes, spines, and image dimensions internally consistent?
5. What image intensity and contrast characteristics should preprocessing account for?
6. What does a representative image look like with its polygon and spine annotations overlaid?

# 1. Imports and configuration

In [ ]:
from pathlib import Path
from collections import Counter
import json
import math
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image, ImageFile
from matplotlib.patches import Polygon, Rectangle

warnings.filterwarnings('ignore', category=FutureWarning)
ImageFile.LOAD_TRUNCATED_IMAGES = True
RANDOM_SEED = 10
rng = np.random.default_rng(RANDOM_SEED)
sns.set_theme(style='whitegrid', context='notebook', palette='muted')
pd.set_option('display.max_columns', 60)
pd.set_option('display.max_rows', 60)
pd.set_option('display.float_format', lambda value: f'{value:.4f}')

WORKSPACE_ROOT = Path.cwd()
DATA_ROOT = WORKSPACE_ROOT / 'MAGFilo_1.0_Kaggle_2026'
TRAIN_DIR = DATA_ROOT / 'train' / 'train_images'
TEST_DIR = DATA_ROOT / 'test' / 'test_images'
ANNOTATION_PATH = DATA_ROOT / 'train' / 'MAGFiLO_1.0_Annotations_kaggle2026_train.json'
print(f'Workspace: {WORKSPACE_ROOT}')
print(f'Dataset:   {DATA_ROOT}')
print(f'JSON:      {ANNOTATION_PATH}')

## Reproducibility and path assumptions

The notebook expects to be opened from the workspace containing `MAGFilo_1.0_Kaggle_2026/`. The path is intentionally relative to the current notebook working directory so the analysis can be moved to another machine or Kaggle with one configuration change.

# 2. Directory inventory

In [ ]:
def describe_path(path):
    if path.is_file():
        return {'name': str(path.relative_to(WORKSPACE_ROOT)), 'type': 'file', 'size_mb': path.stat().st_size / 1024**2, 'children': np.nan}
    return {'name': str(path.relative_to(WORKSPACE_ROOT)), 'type': 'directory', 'size_mb': np.nan, 'children': len(list(path.iterdir()))}

if not DATA_ROOT.exists():
    raise FileNotFoundError(f'Dataset directory not found: {DATA_ROOT}')

inventory = pd.DataFrame([describe_path(path) for path in DATA_ROOT.iterdir()])
display(inventory)

train_files = sorted(TRAIN_DIR.glob('*')) if TRAIN_DIR.exists() else []
test_files = sorted(TEST_DIR.glob('*')) if TEST_DIR.exists() else []
file_inventory = pd.DataFrame({
    'split': ['train', 'test'],
    'directory_exists': [TRAIN_DIR.exists(), TEST_DIR.exists()],
    'files': [len(train_files), len(test_files)],
    'jpeg_files': [sum(path.suffix.lower() in {'.jpg', '.jpeg'} for path in train_files), sum(path.suffix.lower() in {'.jpg', '.jpeg'} for path in test_files)],
})
display(file_inventory)

### Key finding

This first inventory establishes the local evaluation surface. The train and test folder counts should not be silently substituted for COCO image counts because the annotation file can include records whose image files are not included in this workspace.

# 3. Load and validate the COCO-style annotations

In [ ]:
if not ANNOTATION_PATH.exists():
    raise FileNotFoundError(f'Annotation file not found: {ANNOTATION_PATH}')

with ANNOTATION_PATH.open('r', encoding='utf-8') as handle:
    annotations_data = json.load(handle)

required_keys = {'info', 'licenses', 'categories', 'images', 'annotations'}
print('Top-level keys:', list(annotations_data))
print('Missing required keys:', sorted(required_keys - annotations_data.keys()))

categories = pd.DataFrame(annotations_data['categories'])
images = pd.DataFrame(annotations_data['images'])
annotations = pd.DataFrame(annotations_data['annotations'])
print(f'Image records: {len(images):,}')
print(f'Annotation records: {len(annotations):,}')
display(categories)
display(images.head(3))
display(annotations.drop(columns=['segmentation', 'spine'], errors='ignore').head(3))

In [ ]:
schema_summary = pd.DataFrame({
    'table': ['images', 'annotations', 'categories'],
    'rows': [len(images), len(annotations), len(categories)],
    'columns': [', '.join(images.columns), ', '.join(annotations.columns), ', '.join(categories.columns)],
})
display(schema_summary)

print('Image IDs unique:', images['id'].is_unique)
print('Annotation IDs unique:', annotations['id'].is_unique)
print('Annotation image IDs missing from images:', (~annotations['image_id'].isin(images['id'])).sum())
print('Category IDs used but undefined:', sorted(set(annotations['category_id'].dropna()) - set(categories['id'])))
print('Image dimensions:', images[['width', 'height']].drop_duplicates().to_dict('records'))

### Key finding

The annotation format is COCO-like but includes a custom `spine` field. We preserve polygons for segmentation geometry and use spines as a separate structural descriptor. Referential checks here are important because downstream dataset code normally assumes that every annotation points to one valid image and category.

# 4. Match annotation records to local files

In [ ]:
local_files = pd.DataFrame({
    'file_name': [path.name for path in train_files + test_files],
    'split': ['train'] * len(train_files) + ['test'] * len(test_files),
})
images['local_file_name'] = images['file_name'].astype(str).map(lambda value: Path(value).name)
image_file_match = images.merge(local_files, left_on='local_file_name', right_on='file_name', how='left', suffixes=('_annotation', '_local'))
image_file_match['matched_local_file'] = image_file_match['split'].notna()

match_summary = pd.DataFrame({
    'metric': ['COCO image records', 'Local JPEG files', 'Matched annotation records', 'Unmatched annotation records', 'Local files with no annotation record'],
    'value': [len(images), len(local_files), int(image_file_match['matched_local_file'].sum()), int((~image_file_match['matched_local_file']).sum()), int((~local_files['file_name'].isin(images['local_file_name'])).sum())],
})
display(match_summary)
display(image_file_match.loc[~image_file_match['matched_local_file'], ['id', 'file_name', 'local_file_name']].head(10))

### Why this distinction matters

An annotation-level statistic can include records that cannot be visualized locally. Visual inspection and pixel-level image statistics therefore use only matched files, while annotation coverage and schema statistics use the complete JSON.

# 5. Filament and category distribution

In [ ]:
category_names = categories.set_index('id')['name'].to_dict()
annotations['category'] = annotations['category_id'].map(category_names).fillna('Unknown')
annotation_counts = annotations['category'].value_counts().rename_axis('category').reset_index(name='instances')
annotation_counts['percentage'] = annotation_counts['instances'] / len(annotations) * 100
display(annotation_counts)

image_annotation_counts = annotations.groupby('image_id').size().rename('filaments').reset_index()
image_annotation_counts['filaments'].describe()
display(image_annotation_counts['filaments'].describe().to_frame().T)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.barplot(data=annotation_counts, x='category', y='instances', ax=axes[0])
axes[0].set_title('Filament instances by category')
axes[0].set_xlabel('')
axes[0].set_ylabel('Instances')
sns.histplot(data=image_annotation_counts, x='filaments', discrete=True, ax=axes[1])
axes[1].set_title('Filaments per annotated image record')
axes[1].set_xlabel('Number of filaments')
axes[1].set_ylabel('Image records')
plt.tight_layout()
plt.show()

In [ ]:
category_by_image = pd.crosstab(annotations['image_id'], annotations['category'])
category_by_image = category_by_image.reindex(columns=list(category_names.values()), fill_value=0)
category_presence = (category_by_image > 0).sum().sort_values(ascending=False).rename('images').to_frame()
category_presence['percentage_of_image_records'] = category_presence['images'] / len(images) * 100
display(category_presence)

co_occurrence = (category_by_image.gt(0).astype(int).T @ category_by_image.gt(0).astype(int))
plt.figure(figsize=(8, 6))
sns.heatmap(co_occurrence, annot=True, fmt='d', cmap='YlGnBu', square=True, cbar_kws={'label': 'Image records'})
plt.title('Category co-occurrence at image level')
plt.xlabel('Category')
plt.ylabel('Category')
plt.tight_layout()
plt.show()

### Key findings

Instance counts and image-level presence answer different questions. A common category may dominate instance counts while still appearing alongside other categories in multi-filament images. The co-occurrence matrix should guide stratified validation if category balance is important.

# 6. Annotation geometry

In [ ]:
def polygon_points(segmentation):
    if not isinstance(segmentation, list) or len(segmentation) == 0:
        return np.empty((0, 2), dtype=float)
    points = segmentation[0] if isinstance(segmentation[0], list) else segmentation
    if len(points) < 6 or len(points) % 2:
        return np.empty((0, 2), dtype=float)
    return np.asarray(points, dtype=float).reshape(-1, 2)

def polyline_length(values):
    points = np.asarray(values, dtype=float).reshape(-1, 2)
    return float(np.linalg.norm(np.diff(points, axis=0), axis=1).sum()) if len(points) > 1 else 0.0

def polygon_area(points):
    if len(points) < 3:
        return 0.0
    x, y = points[:, 0], points[:, 1]
    return float(abs(np.dot(x, np.roll(y, -1)) - np.dot(y, np.roll(x, -1))) / 2)

geometry_rows = []
for row in annotations.itertuples():
    points = polygon_points(row.segmentation)
    bbox = np.asarray(row.bbox, dtype=float) if isinstance(row.bbox, list) and len(row.bbox) == 4 else np.full(4, np.nan)
    spine = np.asarray(row.spine, dtype=float) if isinstance(row.spine, list) and len(row.spine) % 2 == 0 else np.array([])
    area = polygon_area(points)
    bbox_width, bbox_height = bbox[2], bbox[3]
    geometry_rows.append({
        'id': row.id, 'image_id': row.image_id, 'category': row.category,
        'polygon_vertices': len(points), 'polygon_area': area,
        'bbox_x': bbox[0], 'bbox_y': bbox[1], 'bbox_width': bbox_width, 'bbox_height': bbox_height,
        'bbox_area': bbox_width * bbox_height,
        'bbox_aspect_ratio': bbox_width / bbox_height if bbox_height > 0 else np.nan,
        'polygon_occupancy_in_bbox': area / (bbox_width * bbox_height) if bbox_width > 0 and bbox_height > 0 else np.nan,
        'spine_length': polyline_length(spine) if len(spine) else np.nan,
    })

geometry = pd.DataFrame(geometry_rows)
display(geometry.describe(include='all').T)
display(geometry.groupby('category')[['polygon_area', 'bbox_area', 'bbox_aspect_ratio', 'polygon_occupancy_in_bbox', 'spine_length']].median().sort_values('polygon_area', ascending=False))

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
plot_columns = [('polygon_area', 'Polygon area'), ('bbox_area', 'Bounding-box area'), ('bbox_aspect_ratio', 'Bounding-box aspect ratio'), ('polygon_occupancy_in_bbox', 'Polygon occupancy in bbox'), ('spine_length', 'Spine length'), ('polygon_vertices', 'Polygon vertices')]
for ax, (column, title) in zip(axes.flat, plot_columns):
    sns.histplot(data=geometry, x=column, hue='category', bins=40, element='step', stat='density', common_norm=False, ax=ax)
    ax.set_title(title)
    ax.set_xlabel(column)
    ax.set_ylabel('Density')
plt.tight_layout()
plt.show()

### Interpreting the geometry

Polygon area measures the labelled region, while bounding-box area captures the spatial envelope around it. Low polygon occupancy may indicate thin or curved filaments, which can make downsampling and loss design important. Spine length is not a segmentation target by itself, but it gives a useful structural scale for assessing continuity and morphology.

# 7. Image dimensions and grayscale intensity

In [ ]:
matched_images = image_file_match.loc[image_file_match['matched_local_file'], ['id', 'file_name_annotation', 'local_file_name', 'split']].copy()
path_lookup = {path.name: path for path in train_files + test_files}
sample_names = matched_images['local_file_name'].sample(min(100, len(matched_images)), random_state=RANDOM_SEED).tolist()
intensity_rows = []
for file_name in sample_names:
    path = path_lookup[file_name]
    try:
        with Image.open(path) as image:
            array = np.asarray(image.convert('L'))
        intensity_rows.append({'file_name': file_name, 'width': array.shape[1], 'height': array.shape[0], 'dtype': str(array.dtype), 'mean': array.mean(), 'std': array.std(), 'minimum': array.min(), 'maximum': array.max(), 'p01': np.percentile(array, 1), 'median': np.median(array), 'p99': np.percentile(array, 99)})
    except (OSError, ValueError) as error:
        print(f'Could not read {file_name}: {error}')

image_stats = pd.DataFrame(intensity_rows)
display(image_stats.describe(include='all').T)
display(image_stats[['width', 'height', 'dtype']].value_counts().rename('images').reset_index())

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
sns.histplot(data=image_stats, x='mean', bins=25, ax=axes[0])
axes[0].set_title('Sampled image mean intensity')
sns.histplot(data=image_stats, x='std', bins=25, ax=axes[1])
axes[1].set_title('Sampled image contrast (standard deviation)')
sns.scatterplot(data=image_stats, x='mean', y='std', ax=axes[2])
axes[2].set_title('Mean intensity vs contrast')
plt.tight_layout()
plt.show()

### Key finding

The intensity sample describes acquisition and preprocessing variability without loading every image into memory. The image dimensions should be stable at the competition target resolution; intensity spread and percentile ranges indicate whether per-image normalization or robust contrast handling deserves investigation.

# 8. Data-quality checks

In [ ]:
image_dimensions = images.set_index('id')[['width', 'height']].to_dict('index')
quality_rows = []
for row in annotations.itertuples():
    points = polygon_points(row.segmentation)
    bbox = np.asarray(row.bbox, dtype=float) if isinstance(row.bbox, list) and len(row.bbox) == 4 else np.full(4, np.nan)
    dimensions = image_dimensions.get(row.image_id, {'width': np.nan, 'height': np.nan})
    x_max, y_max = dimensions['width'], dimensions['height']
    polygon_out_of_bounds = len(points) > 0 and (points[:, 0].min() < 0 or points[:, 1].min() < 0 or points[:, 0].max() > x_max or points[:, 1].max() > y_max)
    bbox_out_of_bounds = np.isfinite(bbox).all() and (bbox[0] < 0 or bbox[1] < 0 or bbox[0] + bbox[2] > x_max or bbox[1] + bbox[3] > y_max)
    quality_rows.append({
        'missing_polygon': len(points) == 0,
        'odd_polygon_coordinate_count': isinstance(row.segmentation, list) and len(row.segmentation) > 0 and not isinstance(row.segmentation[0], list) and len(row.segmentation) % 2 != 0,
        'degenerate_polygon': len(points) < 3 or polygon_area(points) == 0,
        'missing_or_invalid_bbox': not np.isfinite(bbox).all() or bbox[2] <= 0 or bbox[3] <= 0,
        'polygon_out_of_bounds': polygon_out_of_bounds,
        'bbox_out_of_bounds': bbox_out_of_bounds,
        'iscrowd_not_zero': row.iscrowd != 0,
        'missing_spine': not isinstance(row.spine, list) or len(row.spine) < 4 or len(row.spine) % 2 != 0,
    })

quality_summary = pd.DataFrame(quality_rows).sum().sort_values(ascending=False).rename('records').to_frame()
quality_summary['percentage'] = quality_summary['records'] / len(annotations) * 100
display(quality_summary)

duplicate_image_names = images[images['local_file_name'].duplicated(keep=False)].sort_values('local_file_name')
print(f'Duplicate image filenames in annotation records: {duplicate_image_names["local_file_name"].nunique()} unique names')
display(duplicate_image_names[['id', 'local_file_name']].head(20))

### Quality interpretation

The checks deliberately separate malformed structures from unusual but valid morphology. A thin filament can have low occupancy without being an error; a polygon with fewer than three vertices, a non-positive box, or coordinates outside the declared image frame is more directly actionable for data-loader and loss-function design.

# 9. Visual inspection of polygon and spine annotations

In [ ]:
def annotations_for_image(image_id):
    return annotations.loc[annotations['image_id'] == image_id]

visual_candidates = matched_images.loc[matched_images['id'].isin(annotations['image_id'])].copy()
visual_sample = visual_candidates.sample(min(6, len(visual_candidates)), random_state=RANDOM_SEED)
fig, axes = plt.subplots(2, 3, figsize=(16, 11))
for ax in axes.flat:
    ax.axis('off')
for ax, record in zip(axes.flat, visual_sample.itertuples()):
    path = path_lookup[record.local_file_name]
    image = np.asarray(Image.open(path).convert('L'))
    ax.imshow(image, cmap='gray')
    image_annotations = annotations_for_image(record.id)
    for annotation in image_annotations.itertuples():
        points = polygon_points(annotation.segmentation)
        if len(points) >= 3:
            ax.add_patch(Polygon(points, closed=True, fill=False, linewidth=1.2, label=annotation.category))
        spine = np.asarray(annotation.spine, dtype=float) if isinstance(annotation.spine, list) and len(annotation.spine) >= 4 else np.array([])
        if len(spine):
            spine_points = spine.reshape(-1, 2)
            ax.plot(spine_points[:, 0], spine_points[:, 1], linewidth=0.8)
    ax.set_title(f'{record.local_file_name}\n{len(image_annotations)} filament(s)')
    ax.set_xlim(0, image.shape[1])
    ax.set_ylim(image.shape[0], 0)
plt.tight_layout()
plt.show()

### Visual reading guide

The polygon boundary shows the supervised segmentation region and the line shows the supplied filament spine. Examine whether thin structures are represented faithfully, whether multiple nearby filaments remain separate, and whether the JPEG background contains artifacts that could be mistaken for filament pixels.

# 10. Image montage without annotations

In [ ]:
montage_sample = local_files.sample(min(12, len(local_files)), random_state=RANDOM_SEED)
fig, axes = plt.subplots(3, 4, figsize=(14, 11))
for ax in axes.flat:
    ax.axis('off')
for ax, record in zip(axes.flat, montage_sample.itertuples()):
    image = np.asarray(Image.open(path_lookup[record.file_name]).convert('L'))
    ax.imshow(image, cmap='gray')
    ax.set_title(f'{record.split}: {record.file_name}', fontsize=8)
plt.tight_layout()
plt.show()

# 11. Consolidated findings and modelling implications

Use the outputs above to record the final measured values after execution. The main conclusions should be stated from the tables and plots rather than assumed in advance.

- **Coverage:** compare COCO image records with local files and decide whether the training pipeline should filter unmatched records explicitly.
- **Labels:** use both instance-level category counts and image-level category presence when designing splits and monitoring class balance.
- **Geometry:** inspect area, occupancy, aspect ratio, and spine length together; thin, elongated objects can be damaged by aggressive resizing.
- **Resolution:** verify the observed dimensions before resizing masks and images, and preserve the same coordinate convention for polygons, boxes, and RLE output.
- **Quality:** treat malformed records as a validation set for the loader, while retaining unusual but valid filament morphology.
- **Preprocessing:** use the sampled intensity distributions and montages to justify grayscale handling, normalization, augmentation, and contrast experiments.
- **Validation:** keep images and all their filament instances together in a split; image-level leakage would make segmentation metrics misleading.

In [ ]:
final_summary = pd.DataFrame({
    'metric': ['COCO image records', 'COCO annotation instances', 'Local train files', 'Local test files', 'Matched annotation image records', 'Distinct categories', 'Sampled readable images'],
    'value': [len(images), len(annotations), len(train_files), len(test_files), int(image_file_match['matched_local_file'].sum()), len(categories), len(image_stats)],
})
display(final_summary)
print('EDA completed successfully.')